## Dependencies

In [ ]:
import ir_datasets
import nltk
import re
import math
import pandas as pd
import numpy as np
import pymongo
import joblib
import numpy as np

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from gensim.models import Word2Vec
from collections import Counter, defaultdict
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import ndcg_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import mysql.connector
from itertools import islice


In [17]:
db_config = {
    "host": "localhost",
    "user": "root",           
    "password": "",  
    "database": "ir_dataset"
}

In [ ]:
dataset = ir_datasets.load("lotte/lifestyle/dev/search")
qrels = dataset.qrels_iter()
queries = dataset.queries_iter()

docs =  dataset.docs_iter()

In [ ]:
def insert_in_batches(cursor, sql, iterator, extract_func, batch_size=10000):
    total_inserted = 0
    while True:
        chunk = list(islice(iterator, batch_size))
        if not chunk:
            break

        data = [extract_func(item) for item in chunk]
        cursor.executemany(sql, data)
        total_inserted += len(data)        
    return total_inserted

try:
    conn = mysql.connector.connect(**db_config)
    cursor = conn.cursor()

    print("Creating tables...")
    
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS queries (
            query_id VARCHAR(255) PRIMARY KEY,
            text TEXT NOT NULL
        )
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS documents (
            doc_id VARCHAR(255) PRIMARY KEY,
            text LONGTEXT NOT NULL
        )
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS qrels (
            query_id VARCHAR(255),
            doc_id VARCHAR(255),
            relevance INT NOT NULL,
            iteration VARCHAR(50),
            PRIMARY KEY (query_id, doc_id),
            FOREIGN KEY (query_id) REFERENCES queries(query_id) ON DELETE CASCADE,
            FOREIGN KEY (doc_id) REFERENCES documents(doc_id) ON DELETE CASCADE
        )
    """)
    conn.commit()

    dataset_name = "lotte/lifestyle/dev/search"
    print(f"Loading dataset: {dataset_name}...")
    dataset = ir_datasets.load(dataset_name)
    
    print("Inserting queries...")
    insert_query_sql = "INSERT IGNORE INTO queries (query_id, text) VALUES (%s, %s)"
    q_count = insert_in_batches(
        cursor=cursor,
        sql=insert_query_sql,
        iterator=dataset.queries_iter(),
        extract_func=lambda q: (q.query_id, q.text)
    )
    print(f"✅ Processed {q_count} queries.")
    conn.commit()

    print("Inserting documents (this might take a moment depending on corpus size)...")
    insert_doc_sql = "INSERT IGNORE INTO documents (doc_id, text) VALUES (%s, %s)"
    d_count = insert_in_batches(
        cursor=cursor,
        sql=insert_doc_sql,
        iterator=dataset.docs_iter(),
        extract_func=lambda d: (d.doc_id, d.text)
    )
    print(f"✅ Processed {d_count} documents.")
    conn.commit()

    print("Inserting qrels...")
    insert_qrel_sql = "INSERT IGNORE INTO qrels (query_id, doc_id, relevance, iteration) VALUES (%s, %s, %s, %s)"
    qr_count = insert_in_batches(
        cursor=cursor,
        sql=insert_qrel_sql,
        iterator=dataset.qrels_iter(),
        extract_func=lambda qr: (qr.query_id, qr.doc_id, qr.relevance, qr.iteration)
    )
    print(f"✅ Processed {qr_count} qrels.")
    conn.commit()

    print("All data successfully stored in MySQL!")

except mysql.connector.Error as err:
    print(f"MySQL Error: {err}")
    if 'conn' in locals() and conn.is_connected():
        conn.rollback()

finally:
    if 'cursor' in locals():
        cursor.close()
    if 'conn' in locals() and conn.is_connected():
        conn.close()
        print("MySQL connection closed.")

Creating tables...
Loading dataset: lotte/lifestyle/dev/search...
Inserting queries...
✅ Processed 417 queries.
Inserting documents (this might take a moment depending on corpus size)...
✅ Processed 268893 documents.
Inserting qrels...
✅ Processed 1376 qrels.
All data successfully stored in MySQL!
MySQL connection closed.


## Pre-processing

In [3]:
class Preprocessor:
    def __init__(self, remove_stopwords=True, stemming=True):
        nltk.download('stopwords')
        self.remove_stopwords = remove_stopwords
        self.stemming = stemming
        self.stop_words = set(stopwords.words("english"))
        self.stemmer = PorterStemmer()

    def normalize(self, text: str) -> str:
        text = text.lower()
        text = re.sub(r"[^a-z0-9\s]", " ", text)  
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def tokenize(self, text: str):
        return text.split()

    def remove_stopwords_fn(self, tokens):
        if not self.remove_stopwords:
            return tokens
        return [t for t in tokens if t not in self.stop_words]

    def stem(self, tokens):
        if not self.stemming:
            return tokens
        return [self.stemmer.stem(t) for t in tokens]

    def process(self, text: str):
        text = self.normalize(text)
        tokens = self.tokenize(text)
        tokens = self.remove_stopwords_fn(tokens)
        tokens = self.stem(tokens)
        return tokens

In [ ]:
try:
    conn = mysql.connector.connect(**db_config)
    cursor_read = conn.cursor(dictionary=True)
    cursor_write = conn.cursor()

    preprocessor = Preprocessor()

    print("Ensuring preprocessed tables exist...")
    
    cursor_write.execute("""
        CREATE TABLE IF NOT EXISTS preprocessed_queries ( 
            query_id VARCHAR(255) PRIMARY KEY,
            text_clean TEXT NOT NULL,
            FOREIGN KEY (query_id) REFERENCES queries(query_id) ON DELETE CASCADE
        )
    """)

    cursor_write.execute("""
        CREATE TABLE IF NOT EXISTS preprocessed_documents (
            doc_id VARCHAR(255) PRIMARY KEY,
            text_clean LONGTEXT NOT NULL,
            FOREIGN KEY (doc_id) REFERENCES documents(doc_id) ON DELETE CASCADE
        )
    """)
    conn.commit()

    def process_table(source_table, target_table, id_col, text_col, batch_size=5000):
        print(f"\nProcessing {source_table} -> {target_table}...")
        
        cursor_read.execute(f"SELECT COUNT(*) as total FROM {source_table}")
        total_rows = cursor_read.fetchone()['total']
        print(f"Total rows to process: {total_rows}")

        offset = 0
        total_inserted = 0
        
        while offset < total_rows:
            cursor_read.execute(f"SELECT {id_col}, {text_col} FROM {source_table} LIMIT %s OFFSET %s", (batch_size, offset))
            rows = cursor_read.fetchall()
            
            if not rows:
                break
            
            processed_data = []
            for row in rows:
                record_id = row[id_col]
                raw_text = row[text_col]
                
                if raw_text:
                    tokens = preprocessor.process(raw_text)
                    clean_text = " ".join(tokens)
                else:
                    clean_text = ""
                    
                processed_data.append((record_id, clean_text))
            
            insert_sql = f"INSERT IGNORE INTO {target_table} ({id_col}, text_clean) VALUES (%s, %s)"
            cursor_write.executemany(insert_sql, processed_data)
            conn.commit()
            
            total_inserted += len(processed_data)
            offset += batch_size
            print(f"  ... Processed {total_inserted} / {total_rows}")

    
    process_table(
        source_table="queries",
        target_table="preprocessed_queries",
        id_col="query_id",
        text_col="text"
    )

    process_table(
        source_table="documents",
        target_table="preprocessed_documents",
        id_col="doc_id",
        text_col="text"
    )

    print("\nAll preprocessing tasks completed successfully!")

except mysql.connector.Error as err:
    print(f"MySQL Error: {err}")
    if 'conn' in locals() and conn.is_connected():
        conn.rollback()

finally:
    if 'cursor_read' in locals(): cursor_read.close()
    if 'cursor_write' in locals(): cursor_write.close()
    if 'conn' in locals() and conn.is_connected():
        conn.close()
        print("MySQL connection closed.")

[nltk_data] Downloading package stopwords to /Users/fadi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Ensuring preprocessed tables exist...

Processing queries -> preprocessed_queries...
Total rows to process: 417
  ... Processed 417 / 417

Processing documents -> preprocessed_documents...
Total rows to process: 268893
  ... Processed 5000 / 268893
  ... Processed 10000 / 268893
  ... Processed 15000 / 268893
  ... Processed 20000 / 268893
  ... Processed 25000 / 268893
  ... Processed 30000 / 268893
  ... Processed 35000 / 268893
  ... Processed 40000 / 268893
  ... Processed 45000 / 268893
  ... Processed 50000 / 268893
  ... Processed 55000 / 268893
  ... Processed 60000 / 268893
  ... Processed 65000 / 268893
  ... Processed 70000 / 268893
  ... Processed 75000 / 268893
  ... Processed 80000 / 268893
  ... Processed 85000 / 268893
  ... Processed 90000 / 268893
  ... Processed 95000 / 268893
  ... Processed 100000 / 268893
  ... Processed 105000 / 268893
  ... Processed 110000 / 268893
  ... Processed 115000 / 268893
  ... Processed 120000 / 268893
  ... Processed 125000 / 268893
 

## TF-IDF

In [ ]:
conn = mysql.connector.connect(**db_config)
docs_df = pd.read_sql("SELECT doc_id, text_clean FROM preprocessed_documents", conn)
conn.close()

doc_ids = docs_df["doc_id"].tolist()
corpus = docs_df["text_clean"].tolist()

vectorizer = TfidfVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b\w+\b"  # assumes already tokenized/cleaned text
)

tfidf_matrix = vectorizer.fit_transform(corpus)

joblib.dump(vectorizer, "tfidf_vectorizer.pkl")
joblib.dump(tfidf_matrix, "tfidf_matrix.pkl")
joblib.dump(doc_ids, "tfidf_doc_ids.pkl")

print("Model saved successfully!")
print("Documents indexed:", len(doc_ids))

/var/folders/rv/pmlk7t214rv1p6hrsl9mzvq40000gn/T/ipykernel_1390/303982869.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  docs_df = pd.read_sql("SELECT doc_id, text_clean FROM preprocessed_documents", conn)


Model saved successfully!
Documents indexed: 268893


In [4]:
class TFIDFSearch:
    def __init__(self):
        self.vectorizer = joblib.load("tfidf_vectorizer.pkl")
        self.tfidf_matrix = joblib.load("tfidf_matrix.pkl")
        self.doc_ids = joblib.load("tfidf_doc_ids.pkl")

    def search(self, query, top_k=10):
        query_vec = self.vectorizer.transform([query])
        scores = cosine_similarity(query_vec, self.tfidf_matrix).flatten()
        top_indices = np.argsort(scores)[::-1][:top_k]
        results = []
        for idx in top_indices:
            results.append(self.doc_ids[idx])
        return results

In [ ]:
tfidf_search = TFIDFSearch()
query = "cold water make lose voic"
results = tfidf_search.search(query, top_k=10)
for r in results:
    print(r)

## Word2Vec

In [5]:
from gensim.models import Word2Vec
import mysql.connector
import pandas as pd
import numpy as np
import joblib

from gensim.models import Word2Vec

In [ ]:
conn = mysql.connector.connect(**db_config)
docs_df = pd.read_sql(
    "SELECT doc_id, text_clean FROM preprocessed_documents",
    conn
)

conn.close()

doc_ids = docs_df["doc_id"].tolist()

tokenized_docs = [
    text.split()
    for text in docs_df["text_clean"]
]

model = Word2Vec(
    sentences=tokenized_docs,
    vector_size=200,
    window=10,
    min_count=3,
    workers=4,
    sg=1,        # skip-gram 
    negative=10,
    epochs=25
)

def document_vector(tokens):
    vectors = [
        model.wv[word]
        for word in tokens
        if word in model.wv
    ]

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

doc_vectors = np.array([
    document_vector(tokens)
    for tokens in tokenized_docs
])

model.save("word2vec_model.model")

joblib.dump(doc_vectors, "word2vec_doc_vectors.pkl")
joblib.dump(doc_ids, "word2vec_doc_ids.pkl")

print("Training completed.")
print("Documents indexed:", len(doc_ids))
print("Vocabulary size:", len(model.wv))

/var/folders/rv/pmlk7t214rv1p6hrsl9mzvq40000gn/T/ipykernel_1390/1371295239.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  docs_df = pd.read_sql(
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: '

Training completed.
Documents indexed: 268893
Vocabulary size: 53805


In [6]:
import numpy as np
import joblib

from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class Word2VecSearch:

    def __init__(self):
        self.model = Word2Vec.load("word2vec_model.model")
        self.doc_vectors = joblib.load(
            "word2vec_doc_vectors.pkl"
        )
        self.doc_ids = joblib.load(
            "word2vec_doc_ids.pkl"
        )

    def query_vector(self, text):
        tokens = text.split()
        vectors = [
            self.model.wv[word]
            for word in tokens
            if word in self.model.wv
        ]
        if len(vectors) == 0:
            return np.zeros(self.model.vector_size)
        return np.mean(vectors, axis=0)

    def search(self, query, top_k=10):
        qvec = self.query_vector(query)
        scores = cosine_similarity(
            [qvec],
            self.doc_vectors
        )[0]
        top_indices = np.argsort(scores)[::-1][:top_k]
        results = []
        for idx in top_indices:
            results.append(self.doc_ids[idx])
        return results


In [ ]:
w2v_search = Word2VecSearch()
results = w2v_search.search(
    "cold water make lose voic",
    top_k=10
)

for r in results:
    print(r)

## BM25

In [8]:
import mysql.connector
import pandas as pd
import joblib
from rank_bm25 import BM25Okapi

In [ ]:

conn = mysql.connector.connect(**db_config)
docs_df = pd.read_sql("SELECT doc_id, text_clean FROM preprocessed_documents", conn)
conn.close()

doc_ids = docs_df["doc_id"].tolist()
corpus = docs_df["text_clean"].tolist()

tokenized_corpus = [doc.split() for doc in corpus]

bm25 = BM25Okapi(tokenized_corpus, k1=1.4, b=0.64)

joblib.dump(bm25, "bm25_model.pkl")
joblib.dump(doc_ids, "bm25_doc_ids.pkl")

print("BM25 model saved successfully!")
print("Documents indexed:", len(doc_ids))

/var/folders/rv/pmlk7t214rv1p6hrsl9mzvq40000gn/T/ipykernel_1390/4134648587.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  docs_df = pd.read_sql("SELECT doc_id, text_clean FROM preprocessed_documents", conn)


BM25 model saved successfully!
Documents indexed: 268893


In [ ]:
class BM25Search:
    def __init__(self):
        self.bm25 = joblib.load("bm25_model.pkl")
        self.doc_ids = joblib.load("bm25_doc_ids.pkl")

    def search(self, query, top_k=10):
        tokens = query.lower().split() 
        scores = self.bm25.get_scores(tokens)
        top_indices = scores.argsort()[::-1][:top_k]
        results = []
        for idx in top_indices:
            results.append(self.doc_ids[idx])
        return results



In [28]:
my_bm25_search = BM25Search()
query = "zebra loach safe shrimp"
results = my_bm25_search.search(query, top_k=10)

for r in results:
    print(r)

7323
3811
6305
619
4573
7794
8586
70166
451
7103


## BERT

In [10]:
import mysql.connector
import pandas as pd
import numpy as np
import joblib
from sentence_transformers import SentenceTransformer


In [ ]:

conn = mysql.connector.connect(**db_config)

docs_df = pd.read_sql(
    "SELECT doc_id, text_clean FROM preprocessed_documents",
    conn
)

conn.close()

doc_ids = docs_df["doc_id"].tolist()
corpus = docs_df["text_clean"].tolist()

model = SentenceTransformer("all-MiniLM-L6-v2")

doc_vectors = model.encode(
    corpus,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

joblib.dump(doc_vectors, "bert_doc_vectors.pkl")
joblib.dump(doc_ids, "bert_doc_ids.pkl")

print("Training completed.")
print("Documents indexed:", len(doc_ids))
print("Embedding dimension:", doc_vectors.shape[1])

/var/folders/rv/pmlk7t214rv1p6hrsl9mzvq40000gn/T/ipykernel_20577/1411018083.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  docs_df = pd.read_sql(
Batches: 100%|██████████| 4202/4202 [39:58<00:00,  1.75it/s]    


Training completed.
Documents indexed: 268893
Embedding dimension: 384


In [ ]:
class BertSearch:

    def __init__(self):
        self.model = SentenceTransformer(
            "all-MiniLM-L6-v2"
        )
        self.doc_vectors = joblib.load(
            "bert_doc_vectors.pkl"
        )
        self.doc_ids = joblib.load(
            "bert_doc_ids.pkl"
        )

    def query_vector(self, text):
        return self.model.encode(
            text,
            normalize_embeddings=True
        )

    def search(self, query, top_k=10):
        qvec = self.query_vector(query)
        scores = cosine_similarity(
            [qvec],
            self.doc_vectors
        )[0]
        top_indices = np.argsort(scores)[::-1][:top_k]
        results = []
        for idx in top_indices:
            results.append(self.doc_ids[idx])
        return results

In [9]:
searcher = BertSearch()
results = searcher.search(
    "zebra loach safe shrimp",
    top_k=10
)
print(results)

['7323', '23561', '4162', '5159', '3811', '4285', '619', '55', '6548', '2763']


## Hybrid representation

In [ ]:
class HybridParallelBM25Word2Vec:
    def __init__(self):
        self.bm25_search = BM25Search()
        self.word2vec_search = Word2VecSearch()
        self.doc_ids = self.bm25_search.doc_ids
    
    def search(self, query, top_k=10):
  
        bm25_tokens = query.lower().split()
        bm25_scores = self.bm25_search.bm25.get_scores(bm25_tokens)
        
        w2v_query_vec = self.word2vec_search.query_vector(query)
        w2v_scores = cosine_similarity(
            [w2v_query_vec],
            self.word2vec_search.doc_vectors
        )[0]
        
        bm25_scores_norm = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min() + 1e-10)
        w2v_scores_norm = (w2v_scores - w2v_scores.min()) / (w2v_scores.max() - w2v_scores.min() + 1e-10)
        
        combined_scores = 0.5 * bm25_scores_norm + 0.5 * w2v_scores_norm
        
        top_indices = np.argsort(combined_scores)[::-1][:top_k]
        results = [self.doc_ids[idx] for idx in top_indices]
        
        return results


class HybridSerialBM25BERT:
    
    def __init__(self, candidate_k=100):
        self.bm25_search = BM25Search()
        self.bert_search = BertSearch()
        self.candidate_k = candidate_k  # Number of candidates from BM25
        self.doc_ids = self.bm25_search.doc_ids
    
    def search(self, query, top_k=10):
        
        bm25_candidates = self.bm25_search.search(query, top_k=self.candidate_k)
        
        query_embedding = self.bert_search.query_vector(query)
        
        candidate_indices = [self.doc_ids.index(doc_id) for doc_id in bm25_candidates]
        candidate_embeddings = self.bert_search.doc_vectors[candidate_indices]
        
        bert_scores = cosine_similarity([query_embedding], candidate_embeddings)[0]
        
        sorted_indices = np.argsort(bert_scores)[::-1][:top_k]
        results = [bm25_candidates[idx] for idx in sorted_indices]
        
        return results


In [ ]:
hybrid_parallel = HybridParallelBM25Word2Vec()
test_query = "zebra loach safe shrimp"
results_parallel = hybrid_parallel.search(test_query, top_k=10)
print("Parallel Hybrid (BM25 + Word2Vec):")
for i, r in enumerate(results_parallel, 1):
    print(f"  {i}. {r}")

print("\n" + "="*50 + "\n")

hybrid_serial = HybridSerialBM25BERT(candidate_k=50)
results_serial = hybrid_serial.search(test_query, top_k=10)
print("Serial Hybrid (BM25 → BERT):")
for i, r in enumerate(results_serial, 1):
    print(f"  {i}. {r}")


Parallel Hybrid (BM25 + Word2Vec):
  1. 7323
  2. 3811
  3. 6305
  4. 619
  5. 4573
  6. 8586
  7. 7794
  8. 70166
  9. 451
  10. 7103


Serial Hybrid (BM25 → BERT):
  1. 7323
  2. 23561
  3. 4162
  4. 5159
  5. 3811
  6. 4285
  7. 619
  8. 6548
  9. 2763
  10. 2053


## Evaluation

In [15]:
import mysql.connector
import math
from collections import defaultdict

In [ ]:
tfidf_search = TFIDFSearch()
word2vec_search = Word2VecSearch()
bm25_search = BM25Search()
bert_search = BertSearch()
hybrid_parallel = HybridParallelBM25Word2Vec()
hybrid_serial = HybridSerialBM25BERT(candidate_k=50)


def compute_query_metrics(ranked_list: list, qrel_list: list, top_k_p: int = 10) -> dict:
    total_relevant = sum(1 for rel in qrel_list)
    
    if total_relevant == 0:
        return None 

    top_k_docs = ranked_list[:top_k_p]
    rel_in_top_k = sum(1 for doc_id in top_k_docs if doc_id in qrel_list)
    precision_at_k = rel_in_top_k / top_k_p

    rel_retrieved = sum(1 for doc_id in ranked_list if doc_id in qrel_list)
    recall = rel_retrieved / total_relevant

    ap_sum = 0.0
    num_rel_found = 0
    for rank, doc_id in enumerate(ranked_list, start=1):
        if doc_id in qrel_list:
            num_rel_found += 1
            precision_at_rank = num_rel_found / rank
            ap_sum += precision_at_rank
    ap = ap_sum / total_relevant

    dcg = 0.0
    for rank, doc_id in enumerate(ranked_list, start=1):
        rel = 1 if doc_id in qrel_list else 0
        dcg += (2**rel - 1) / math.log2(rank + 1)

    ideal_rels = sorted([1 if doc_id in qrel_list else 0 for doc_id in ranked_list], reverse=True)
    idcg = 0.0
    for rank, rel in enumerate(ideal_rels[:len(ranked_list)], start=1):
        idcg += (2**rel - 1) / math.log2(rank + 1)

    ndcg = dcg / idcg if idcg > 0 else 0.0

    return {
        "p_at_k": precision_at_k,
        "recall": recall,
        "ap": ap,
        "ndcg": ndcg
    }


try:
    conn = mysql.connector.connect(**db_config)
    cursor = conn.cursor(dictionary=True)

    print("Loading qrels from database...")
    cursor.execute("SELECT query_id, doc_id, relevance FROM qrels")
    
    qrels_lookup = defaultdict(list)
    for row in cursor.fetchall():
        qrels_lookup[row['query_id']].append(row['doc_id'])

    print("Loading preprocessed queries...")
    cursor.execute("SELECT query_id, text_clean FROM preprocessed_queries")
    queries = cursor.fetchall()
    
    print(f"Loaded {len(queries)} queries and {len(qrels_lookup)} qrel mappings.\n")

    models_to_evaluate = {
        "TF-IDF": tfidf_search.search,
        "Word2Vec": word2vec_search.search,
        "BM25": bm25_search.search,
        "BERT": bert_search.search,
        "Parallel (BM25+W2V)": hybrid_parallel.search,
        "Serial (BM25→BERT)": hybrid_serial.search
    }

    final_results = {}

    for model_name, retrieve_fn in models_to_evaluate.items():
        print(f"Running evaluation for model: {model_name}...")
        
        total_p_at_10 = 0.0
        total_recall = 0.0
        total_ap = 0.0
        total_ndcg = 0.0
        evaluated_queries_count = 0

        for q in queries:
            q_id = q['query_id']
            q_text = q['text_clean']

            ranked_results = retrieve_fn(query=q_text, top_k=100)
            
            if not ranked_results:
                ranked_results = []

            metrics = compute_query_metrics(ranked_results, qrels_lookup[q_id], top_k_p=10)
            
            if metrics:
                total_p_at_10 += metrics["p_at_k"]
                total_recall += metrics["recall"]
                total_ap += metrics["ap"]
                total_ndcg += metrics["ndcg"]
                evaluated_queries_count += 1

        if evaluated_queries_count > 0:
            final_results[model_name] = {
                "MAP": total_ap / evaluated_queries_count,
                "Recall": total_recall / evaluated_queries_count,
                "Precision@10": total_p_at_10 / evaluated_queries_count,
                "nDCG": total_ndcg / evaluated_queries_count
            }
        else:
            final_results[model_name] = {"MAP": 0.0, "Recall": 0.0, "Precision@10": 0.0, "nDCG": 0.0}

    print("\n======================= EVALUATION REPORT =======================")
    print(f"{'Model':<20} | {'MAP':<10} | {'Recall':<10} | {'Precision@10':<12} | {'nDCG':<10}")
    print("-" * 75)
    for model_name, metrics in final_results.items():
        print(f"{model_name:<20} | {metrics['MAP']:<10.4f} | {metrics['Recall']:<10.4f} | {metrics['Precision@10']:<12.4f} | {metrics['nDCG']:<10.4f}")
    print("===================================================================")

except mysql.connector.Error as err:
    print(f"MySQL Error: {err}")

finally:
    if 'cursor' in locals(): cursor.close()
    if 'conn' in locals() and conn.is_connected():
        conn.close()


Loading qrels from database...
Loading preprocessed queries...
Loaded 417 queries and 417 qrel mappings.

Running evaluation for model: TF-IDF...
Running evaluation for model: Word2Vec...
Running evaluation for model: BM25...
Running evaluation for model: BERT...
Running evaluation for model: Parallel (BM25+W2V)...
Running evaluation for model: Serial (BM25→BERT)...

======================= EVALUATION REPORT =======================
Model                | MAP        | Recall     | Precision@10 | nDCG      
---------------------------------------------------------------------------
TF-IDF               | 0.1904     | 0.5341     | 0.0813       | 0.3772    
Word2Vec             | 0.1568     | 0.5385     | 0.0741       | 0.3419    
BM25                 | 0.2822     | 0.6433     | 0.1086       | 0.4879    
BERT                 | 0.1667     | 0.5574     | 0.0748       | 0.3587    
Parallel (BM25+W2V)  | 0.2765     | 0.6549     | 0.1108       | 0.4820    
Serial (BM25→BERT)   | 0.2162     | 0.